# Notebook 02 — Geospatial Data Quality
Validates the geolocation table: nulls, out-of-bounds coordinates, and ZIP duplicates.

In [ ]:
import pathlib
import matplotlib.pyplot as plt
import pandas as pd
from src.quality_checks import null_report, detect_out_of_bounds, aggregate_centroids, geo_quality_report
import src.quality_checks
import importlib
import sys
sys.path.insert(0, '..')

importlib.reload(src.quality_checks)


OUTPUTS = pathlib.Path('..') / 'outputs'
print('Imports OK')

Imports OK


In [2]:
geo = pd.read_csv(OUTPUTS / 'processed_data' / 'olist_geolocation_dataset.csv')
print(f'Rows: {len(geo):,}  |  Columns: {list(geo.columns)}')

Rows: 1,000,163  |  Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


## 1. Null Report

In [ ]:
geo_cols = ['geolocation_lat', 'geolocation_lng',
            'geolocation_zip_code_prefix']
display(null_report(geo, geo_cols))

,null_count,null_pct
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_zip_code_prefix,0,0.0


## 2. Out-of-Bounds Coordinates

In [4]:
oob = detect_out_of_bounds(geo)
print(f'Out-of-bounds rows: {len(oob):,}  ({len(oob)/len(geo)*100:.2f}%)')
display(oob.head())

Out-of-bounds rows: 31  (0.00%)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df, title in zip(axes, [geo, oob], ['All coordinates', 'Out-of-bounds only']):
    ax.scatter(df['geolocation_lng'], df['geolocation_lat'],
               s=0.1, alpha=0.3, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 3. ZIP Duplicates & Centroid Aggregation

In [5]:
dup_count = geo.duplicated(subset=['geolocation_zip_code_prefix']).sum()
print(f'Duplicate ZIP rows: {dup_count:,}')
print(f'Unique ZIPs:        {geo["geolocation_zip_code_prefix"].nunique():,}')

Duplicate ZIP rows: 981,148
Unique ZIPs:        19,015


In [6]:
centroids = aggregate_centroids(geo)
print(f'Centroids computed: {len(centroids):,}')
display(centroids.head())

Centroids computed: 19,011


,geolocation_zip_code_prefix,lat,lng
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [7]:
centroids.to_csv(OUTPUTS / 'processed_data' / 'geo_centroids.csv', index=False)
print('geo_centroids.csv saved.')

geo_centroids.csv saved.


## 4. Quality Report

In [8]:
report = geo_quality_report(geo)
display(report)
report.to_csv(OUTPUTS / 'reports' / 'geo_quality_report.csv', index=False)
print('geo_quality_report.csv saved.')

,total_rows,out_of_bounds,out_of_bounds_pct,duplicate_zips,null_coords
0,1000163,31,0.0,981148,0


geo_quality_report.csv saved.
